# Chatterbox voice server on a free Colab GPU
Makes your Studio voices much more human (English, Hindi, Hinglish) while your Mac does everything else.

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run the cells in order. Cell 3 asks you to upload `scripts/chatterbox-server.py` from the project.
3. The last cell prints two lines. Paste them into `apps/web/.env.local` and `apps/voice/.env.local` on your Mac, then restart the web app and the voice service.

Keep this tab open while you use it; Colab stops it after a while of inactivity. Your staff recordings are sent here when a voice is first used: use your own Google account and only voices whose owners agreed. This is for testing, not for client calls.

In [ ]:
!nvidia-smi | head -15

In [ ]:
!pip -q install chatterbox-tts==0.1.7

In [ ]:
from google.colab import files
up = files.upload()   # choose scripts/chatterbox-server.py
assert 'chatterbox-server.py' in up, 'upload chatterbox-server.py'

In [ ]:
import os, secrets, subprocess, time, urllib.request
TOKEN = secrets.token_urlsafe(32)
os.makedirs('/content/voices', exist_ok=True)
env = dict(os.environ, CHATTERBOX_TOKEN=TOKEN, CHATTERBOX_HOST='127.0.0.1', CHATTERBOX_PORT='8771', POCKET_TTS_VOICES_DIR='/content/voices')
server = subprocess.Popen(['python', '/content/chatterbox-server.py'], env=env, stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
for i in range(120):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8771/health', timeout=2).status == 200: print('server ready'); break
    except Exception: time.sleep(5)
else:
    print(open('/content/server.log').read()[-2000:])

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import re
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8771'], stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for i in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: url = m.group(0); break
    time.sleep(2)
print()
print('Paste these two lines into apps/web/.env.local AND apps/voice/.env.local:')
print()
print(f'CHATTERBOX_URL={url}')
print(f'CHATTERBOX_TOKEN={TOKEN}')